Cell 1: Splitting Data & Representasi Vektor (Poin i & ii)

Langkah pertama adalah membagi 30 kasusmu menjadi data latih (train) sebagai case base utama, dan data uji (test) untuk pengujian nanti, lalu mengubah teksnya menjadi angka (vektor).

In [2]:
import os
import json
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Konfigurasi Folder
os.makedirs("../data/eval", exist_ok=True)

# Load Data Processed dari Tahap 2
df = pd.read_csv("../data/processed/cases.csv")

# Menggabungkan ringkasan dan teks full untuk basis pencarian yang lebih kaya
df['teks_pencarian'] = df['ringkasan_fakta'].fillna('') + " " + df['text_full'].fillna('')

print("Memulai Tahap 3: Case Retrieval...")


# ii. Splitting Data
# Rasio 80:20 (80% Train/Case Base, 20% Test/Kasus Baru)
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
print(f"Data Splitting Selesai! Train: {len(df_train)} kasus, Test: {len(df_test)} kasus.")


# i. Representasi Vektor (Pendekatan 1: TF-IDF)
vectorizer = TfidfVectorizer(max_features=5000)

# Fit dan transform HANYA pada data train (sebagai database kasus lama)
tfidf_matrix_train = vectorizer.fit_transform(df_train['teks_pencarian'])
print("Representasi Vektor (TF-IDF) berhasil dibuat!")

Memulai Tahap 3: Case Retrieval...
Data Splitting Selesai! Train: 24 kasus, Test: 6 kasus.
Representasi Vektor (TF-IDF) berhasil dibuat!


Cell 2: Fungsi Retrieval (Poin iii & iv)

In [3]:
from sklearn.naive_bayes import MultinomialNB


# iii.1. Model Retrieval menggunakan Naive Bayes
print("Melatih model Naive Bayes untuk Retrieval...")

# Kita jadikan 'case_id' sebagai label/target klasifikasi
y_train = df_train['case_id']

# Membuat dan melatih model Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(tfidf_matrix_train, y_train)


# iv. Fungsi Retrieval (Revisi Menggunakan Model ML)
def retrieve(query: str, k: int = 5) -> list:
    """
    Mencari top-k kasus lama menggunakan probabilitas prediksi Naive Bayes.
    """
    # 1) Pre-process & Hitung vektor query
    query_vec = vectorizer.transform([query])
    
    # 2) Dapatkan probabilitas query ini masuk ke masing-masing 'kelas' (case_id)
    # predict_proba mengembalikan array 2D, kita ambil baris pertamanya [0]
    probabilitas = nb_model.predict_proba(query_vec)[0]
    
    # 3) Dapatkan top-k index berdasarkan nilai probabilitas tertinggi
    top_k_indices = probabilitas.argsort()[-k:][::-1]
    
    hasil_retrieval = []
    for idx in top_k_indices:
        # Ambil nama case_id dari atribut classes_ model
        case_id_prediksi = nb_model.classes_[idx]
        
        # Cari detail kasusnya di df_train
        detail_kasus = df_train[df_train['case_id'] == case_id_prediksi].iloc[0]
        
        hasil_retrieval.append({
            'case_id': str(detail_kasus['case_id']),
            'no_perkara': str(detail_kasus['no_perkara']),
            'similarity_score': float(probabilitas[idx]), # Sekarang ini adalah probabilitas klasifikasi
            'amar_putusan': str(detail_kasus['amar_putusan'])
        })
        
    return [kasus['case_id'] for kasus in hasil_retrieval], hasil_retrieval

# Uji coba singkat fungsi yang baru
contoh_query = "Kasus pelanggaran merek pakaian olahraga dengan logo dan nama yang menyerupai merek terkenal"
top_ids, detail_hasil = retrieve(contoh_query, k=3)

print("Uji Coba Fungsi retrieve() dengan Naive Bayes:")
print(f"Query: '{contoh_query}'")
print(f"Top-3 Case ID yang mirip: {top_ids}")

Melatih model Naive Bayes untuk Retrieval...
Uji Coba Fungsi retrieve() dengan Naive Bayes:
Query: 'Kasus pelanggaran merek pakaian olahraga dengan logo dan nama yang menyerupai merek terkenal'
Top-3 Case ID yang mirip: ['case_004', 'case_012', 'case_014']


Cell 3: Pengujian Awal & Output (Poin v & vi)

In [4]:
import json
import os


# v. Pengujian Awal & vi. Output
print("Menyiapkan data pengujian (queries.json)...")

# Membuat folder eval jika belum ada
folder_eval = "../data/eval"
os.makedirs(folder_eval, exist_ok=True)

queries_uji = []

# Mengambil 5 kasus dari data test (df_test) sebagai query uji
for index, baris in df_test.head(5).iterrows():
    # Menggunakan ringkasan fakta sebagai query
    # Kita batasi sekitar 500 karakter untuk mensimulasikan input cerita kasus baru
    teks_mentah = str(baris['ringkasan_fakta']) if baris['ringkasan_fakta'] != "TIDAK DITEMUKAN" else str(baris['text_full'])
    teks_query = teks_mentah[:500] + "..." 
    
    queries_uji.append({
        "query_id": f"query_{baris['case_id']}",
        "query_text": teks_query,
        "ground_truth_case_id": baris['case_id'] # Ini adalah kunci jawaban aslinya
    })

# Menyimpan ke dalam file JSON
path_queries = os.path.join(folder_eval, "queries.json")
with open(path_queries, "w", encoding="utf-8") as f:
    json.dump(queries_uji, f, indent=4)

print("-" * 60)
print("Tahap 3 Selesai Sepenuhnya!")
print(f"5 Query uji berhasil disimpan di: {path_queries}")
print("Fungsi retrieve() sudah teruji dengan model Naive Bayes.")
print("-" * 60)

Menyiapkan data pengujian (queries.json)...
------------------------------------------------------------
Tahap 3 Selesai Sepenuhnya!
5 Query uji berhasil disimpan di: ../data/eval\queries.json
Fungsi retrieve() sudah teruji dengan model Naive Bayes.
------------------------------------------------------------
